# RAG Pipeline — LangGraph + Google Gemini (FREE) + FAISS

**100% Free Setup:**
- LLM: Google Gemini 2.5 Flash — free tier, no credit card needed
- Embeddings: `sentence-transformers` — runs locally, zero API cost
- Vector Store: FAISS — fully local
- Token Mapper Agent: tracks real-time token usage per node

Get a FREE Gemini API key (30 seconds, no credit card): https://aistudio.google.com/apikey

---

## Cell 0 — Install Dependencies

Run this cell **once** to install all packages into the active Jupyter kernel's Python.

In [ ]:
import sys
print(f"Installing into: {sys.executable}")

packages = [
    "python-dotenv",
    "sentence-transformers",
    "faiss-cpu",
    "pypdf",
    "google-genai",
    "langgraph",
    "langchain",
    "langchain-core",
    "langchain-community",
    "pandas",
    "numpy",
    "tabulate",
    "tqdm",
]

import subprocess
result = subprocess.run(
    [sys.executable, "-m", "pip", "install"] + packages + ["-q"],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("All packages installed successfully!")
else:
    print("STDOUT:", result.stdout[-2000:] if result.stdout else "")
    print("STDERR:", result.stderr[-2000:] if result.stderr else "")

## Cell 1 — Imports & Setup

In [ ]:
import os, time, warnings
from pathlib import Path
from typing import TypedDict, List, Any
from dataclasses import dataclass

warnings.filterwarnings("ignore")

from dotenv import load_dotenv
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")
if not GEMINI_API_KEY or GEMINI_API_KEY == "your_gemini_api_key_here":
    raise EnvironmentError(
        "GEMINI_API_KEY not set!\n"
        "  1. Go to: https://aistudio.google.com/apikey\n"
        "  2. Click 'Create API Key' (free, no credit card)\n"
        "  3. Add to .env:  GEMINI_API_KEY=AIza..."
    )
print(f"Gemini API key loaded (...{GEMINI_API_KEY[-6:]})")

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from google import genai
from google.genai import types as genai_types
from langgraph.graph import StateGraph, END
import pandas as pd
from IPython.display import display, Markdown, HTML

print(f"Python: {os.sys.version}")
print("All imports successful!")

## Cell 2 — Configuration

> Edit these settings before running.

In [ ]:
# ── EDIT THESE ─────────────────────────────────────────────────

PDF_PATH = "GRE_score.pdf"   # <- your PDF filename here

# Gemini model — all are FREE tier
# "gemini-2.5-flash"         <- recommended (latest, smartest)
# "gemini-2.0-flash-001"     <- stable versioned release
# "gemini-2.0-flash-lite"    <- fastest / lightest
GEMINI_MODEL = "gemini-2.5-flash"

CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
TOP_K           = 4
EMBEDDING_MODEL = "all-MiniLM-L6-v2"

# ───────────────────────────────────────────────────────────────
gemini_client = genai.Client(api_key=GEMINI_API_KEY)

print(f"PDF Path:          {PDF_PATH}")
print(f"Gemini Model:      {GEMINI_MODEL}  [FREE TIER]")
print(f"Chunk Size:        {CHUNK_SIZE} chars  (overlap: {CHUNK_OVERLAP})")
print(f"Top-K Retrieval:   {TOP_K} chunks")
print(f"Embedding Model:   {EMBEDDING_MODEL}  (local / offline)")

## Cell 3 — Token Mapper Agent (Agent 2)

In [ ]:
@dataclass
class NodeTokenRecord:
    node_name:     str
    input_tokens:  int   = 0
    output_tokens: int   = 0
    elapsed_ms:    float = 0.0

    @property
    def total_tokens(self):
        return self.input_tokens + self.output_tokens


class TokenMapperAgent:
    """
    Agent 2 — Real-Time Token Mapper.
    Hooks into every LangGraph node to record token usage and latency.
    Gemini free tier: cost is always $0.00.
    """
    def __init__(self):
        self.records: List[NodeTokenRecord] = []
        self._timers = {}

    def start_node(self, name: str):
        self._timers[name] = time.perf_counter()
        print(f"  --> [{name}] starting...")

    def end_node(self, name: str, input_tokens=0, output_tokens=0):
        elapsed = 0.0
        if name in self._timers:
            elapsed = (time.perf_counter() - self._timers.pop(name)) * 1000
        self.records.append(NodeTokenRecord(
            node_name=name, input_tokens=input_tokens,
            output_tokens=output_tokens, elapsed_ms=elapsed,
        ))
        tok = f"{input_tokens} in / {output_tokens} out" if (input_tokens + output_tokens) else "(no LLM call)"
        print(f"  <-- [{name}] done  |  tokens: {tok}  |  {elapsed:.0f} ms")

    def report(self):
        if not self.records:
            print("No records yet."); return
        rows, ti, to, tm = [], 0, 0, 0.0
        for r in self.records:
            ti += r.input_tokens; to += r.output_tokens; tm += r.elapsed_ms
            rows.append({
                "Node": r.node_name,
                "Input Tokens":  r.input_tokens  if r.input_tokens  else "—",
                "Output Tokens": r.output_tokens if r.output_tokens else "—",
                "Total Tokens":  r.total_tokens  if r.total_tokens  else "—",
                "Time (ms)": f"{r.elapsed_ms:.0f}",
                "Cost": "$0.00 [FREE]",
            })
        rows.append({"Node": "TOTAL", "Input Tokens": ti, "Output Tokens": to,
                     "Total Tokens": ti + to, "Time (ms)": f"{tm:.0f}",
                     "Cost": "$0.00 [FREE TIER]"})
        df = pd.DataFrame(rows)
        display(Markdown("### Token Mapper Agent — Per-Node Usage"))
        display(HTML(
            df.to_html(index=False, border=0)
            .replace('<table', '<table style="border-collapse:collapse;font-family:monospace;width:100%;"')
            .replace('<th>', '<th style="background:#1e3a5f;color:#e2e8f0;padding:8px 14px;border:1px solid #2d4a7a;text-align:left;">')
            .replace('<td>', '<td style="padding:7px 14px;border:1px solid #2d4a7a;">')
        ))

    def reset(self):
        self.records.clear(); self._timers.clear()


token_agent = TokenMapperAgent()
print("Token Mapper Agent ready.")

## Cell 4 — RAG Pipeline Nodes (Agent 1)

In [ ]:
class PipelineState(TypedDict, total=False):
    pdf_path: str; question: str; chunks: List[str]; chunk_meta: List[dict]
    embeddings: Any; faiss_index: Any; retrieved: List[str]
    retrieved_meta: List[dict]; answer: str; token_usage: dict

def _chunk_text(text, size, overlap):
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start: start + size]); start += size - overlap
    return [c.strip() for c in chunks if c.strip()]

print(f"Loading embedding model '{EMBEDDING_MODEL}' (local, free)...")
embedder = SentenceTransformer(EMBEDDING_MODEL)
print("Embedding model ready.")


def ingest_pdf(state: PipelineState) -> PipelineState:
    token_agent.start_node("ingest_pdf")
    if not Path(state["pdf_path"]).exists():
        raise FileNotFoundError(f"PDF not found: {state['pdf_path']}")
    reader = PdfReader(state["pdf_path"])
    chunks, meta = [], []
    for page_num, page in enumerate(reader.pages, 1):
        text = page.extract_text() or ""
        for i, chunk in enumerate(_chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)):
            chunks.append(chunk); meta.append({"page": page_num, "chunk_index": i})
    token_agent.end_node("ingest_pdf")
    print(f"     {len(reader.pages)} pages -> {len(chunks)} chunks")
    return {**state, "chunks": chunks, "chunk_meta": meta}


def embed_and_index(state: PipelineState) -> PipelineState:
    token_agent.start_node("embed_and_index")
    chunks = state["chunks"]
    print(f"     Embedding {len(chunks)} chunks locally (free, no API)...")
    emb = embedder.encode(chunks, batch_size=64, show_progress_bar=True,
                          convert_to_numpy=True, normalize_embeddings=True).astype("float32")
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(emb)
    token_agent.end_node("embed_and_index")
    print(f"     FAISS index: {index.ntotal} vectors, dim={emb.shape[1]}")
    return {**state, "embeddings": emb, "faiss_index": index}


def retrieve(state: PipelineState) -> PipelineState:
    token_agent.start_node("retrieve")
    q_emb = embedder.encode([state["question"]], convert_to_numpy=True,
                            normalize_embeddings=True).astype("float32")
    scores, indices = state["faiss_index"].search(q_emb, TOP_K)
    retrieved      = [state["chunks"][i]     for i in indices[0] if i >= 0]
    retrieved_meta = [state["chunk_meta"][i] for i in indices[0] if i >= 0]
    print(f"     Top-{TOP_K} scores: [{', '.join(f'{s:.3f}' for s in scores[0])}]")
    for m in retrieved_meta:
        print(f"       -> Page {m['page']}, chunk #{m['chunk_index']}")
    token_agent.end_node("retrieve")
    return {**state, "retrieved": retrieved, "retrieved_meta": retrieved_meta}


def generate_answer(state: PipelineState) -> PipelineState:
    token_agent.start_node("generate_answer")
    context = "\n\n---\n\n".join(
        f"[Chunk {i+1} | Page {m['page']}]\n{chunk}"
        for i, (chunk, m) in enumerate(zip(state["retrieved"], state["retrieved_meta"]))
    )
    prompt = (
        "You are a precise document assistant. "
        "Answer ONLY using the context below. "
        "If not found say: 'This information is not in the document.' "
        "Cite the chunk number(s) used.\n\n"
        f"CONTEXT:\n{context}\n\nQUESTION: {state['question']}\n\nANSWER:"
    )
    response = gemini_client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config=genai_types.GenerateContentConfig(temperature=0.2, max_output_tokens=1024),
    )
    answer = response.text.strip()
    usage  = response.usage_metadata
    token_usage = {
        "input_tokens":  usage.prompt_token_count     or 0,
        "output_tokens": usage.candidates_token_count or 0,
        "total_tokens":  usage.total_token_count      or 0,
    }
    token_agent.end_node("generate_answer",
                         input_tokens=token_usage["input_tokens"],
                         output_tokens=token_usage["output_tokens"])
    return {**state, "answer": answer, "token_usage": token_usage}


def token_summary(state: PipelineState) -> PipelineState:
    token_agent.start_node("token_summary")
    token_agent.end_node("token_summary")
    return state


print("All 5 pipeline nodes defined.")

## Cell 5 — Build & Compile the LangGraph

In [ ]:
builder = StateGraph(PipelineState)
builder.add_node("ingest_pdf",      ingest_pdf)
builder.add_node("embed_and_index", embed_and_index)
builder.add_node("retrieve",        retrieve)
builder.add_node("generate_answer", generate_answer)
builder.add_node("token_summary",   token_summary)
builder.set_entry_point("ingest_pdf")
builder.add_edge("ingest_pdf",      "embed_and_index")
builder.add_edge("embed_and_index", "retrieve")
builder.add_edge("retrieve",        "generate_answer")
builder.add_edge("generate_answer", "token_summary")
builder.add_edge("token_summary",   END)
pipeline = builder.compile()

print("LangGraph pipeline compiled!")
print("  ingest_pdf  ->  embed_and_index  ->  retrieve  ->  generate_answer  ->  token_summary  ->  END")

## Cell 6 — Run the Pipeline

> Edit `QUESTION` below then run this cell.

In [ ]:
QUESTION = "What were the Verbal Reasoning, Quantitative Reasoning, and Analytical Writing scores?"

token_agent.reset()
print("=" * 60)
print(f"  PDF:      {PDF_PATH}")
print(f"  Question: {QUESTION}")
print(f"  Model:    {GEMINI_MODEL}  [FREE TIER]")
print("=" * 60)

final_state = pipeline.invoke({"pdf_path": PDF_PATH, "question": QUESTION})

print()
print("=" * 60)
print("  ANSWER")
print("=" * 60)
display(Markdown(final_state["answer"]))

print()
print("=" * 60)
print("  RETRIEVED SOURCE CHUNKS")
print("=" * 60)
for i, (chunk, meta) in enumerate(zip(final_state["retrieved"], final_state["retrieved_meta"]), 1):
    print(f"\n[Chunk {i}] Page {meta['page']}, chunk #{meta['chunk_index']}")
    print("-" * 40)
    print(chunk[:300] + ("..." if len(chunk) > 300 else ""))

## Cell 7 — Token Mapper Agent Report

In [ ]:
token_agent.report()

if "token_usage" in final_state:
    u = final_state["token_usage"]
    print()
    print("Raw Gemini token counts:")
    print(f"  Prompt tokens:    {u['input_tokens']:,}")
    print(f"  Candidate tokens: {u['output_tokens']:,}")
    print(f"  Total tokens:     {u['total_tokens']:,}")
    print()
    print("Free tier limits (Gemini 2.5 Flash):")
    print("  500 requests/day | 1,000,000 tokens/day | $0.00")

## Cell 8 — Follow-up Questions (no re-embedding)

In [ ]:
def ask(question: str, existing_state: PipelineState):
    """Ask a follow-up without re-ingesting or re-embedding the PDF."""
    mini = StateGraph(PipelineState)
    mini.add_node("retrieve",        retrieve)
    mini.add_node("generate_answer", generate_answer)
    mini.add_node("token_summary",   token_summary)
    mini.set_entry_point("retrieve")
    mini.add_edge("retrieve",        "generate_answer")
    mini.add_edge("generate_answer", "token_summary")
    mini.add_edge("token_summary",   END)
    token_agent.reset()
    print(f"Question: {question}\n")
    result = mini.compile().invoke({**existing_state, "question": question})
    display(Markdown("**Answer:**\n\n" + result["answer"]))
    token_agent.report()
    return result


# Uncomment any line below:
# final_state = ask("What scores are mentioned in the document?", final_state)
# final_state = ask("What test sections are covered?", final_state)
# final_state = ask("Summarize all the scores.", final_state)

print("Follow-up helper ready. Uncomment a line above to use it.")